In [26]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # 完全隐藏 GPU，强制纯 CPU
from MetaQ_sc import run_metaq
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scanpy as sc
from umap import UMAP
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import anndata as ad
import stark as sk


In [27]:
lb = []
path = '/Users/ckw/warehouse/metacell/data/test_700_snm3c'
for val in os.listdir(path):
    if val.endswith('.pairs'):
        lb.append(val.split('.pairs')[0].split('_')[1])
lb = ['ExcNeuron' if x in ['L23', 'L4', 'L5', 'L6'] else x for x in lb]
pca_vec = np.load('/Users/ckw/warehouse/metacell/stark/test_output/pca_vec_500000.npy')
umap_vec = np.load('/Users/ckw/warehouse/metacell/stark/test_output/umap_vec_500000.npy')
cell_embeddings = pca_vec  # 或者 umap_vec，取决于你想用哪个作为输入
print(pca_vec.shape, umap_vec.shape, len(lb))
tmp_val = pca_vec.copy()
min_val = np.min(tmp_val)
tmp_val -= min_val
adata1 = ad.AnnData(tmp_val)
adata1.obs['cell_type'] = lb
adata1.obsm['X_pca'] = pca_vec
adata1.obsm['X_umap'] = umap_vec

(700, 114) (700, 2) 700


In [28]:
lb = []
path = '/Users/ckw/warehouse/metacell/data/test_700_snm3c'
for val in os.listdir(path):
    if val.endswith('.pairs'):
        lb.append(val.split('.pairs')[0].split('_')[1])
lb = ['ExcNeuron' if x in ['L23', 'L4', 'L5', 'L6'] else x for x in lb]
pca_vec = np.load('/Users/ckw/warehouse/metacell/stark/test_output/pca_vec_50000.npy')
umap_vec = np.load('/Users/ckw/warehouse/metacell/stark/test_output/umap_vec_50000.npy')
cell_embeddings = pca_vec  # 或者 umap_vec，取决于你想用哪个作为输入
print(pca_vec.shape, umap_vec.shape, len(lb))
tmp_val = pca_vec.copy()
min_val = np.min(tmp_val)
tmp_val -= min_val
adata2 = ad.AnnData(tmp_val)
adata2.obs['cell_type'] = lb
adata2.obsm['X_pca'] = pca_vec
adata2.obsm['X_umap'] = umap_vec

(700, 68) (700, 2) 700


In [29]:
lb = []
path = '/Users/ckw/warehouse/metacell/data/test_700_snm3c'
for val in os.listdir(path):
    if val.endswith('.pairs'):
        lb.append(val.split('.pairs')[0].split('_')[1])
lb = ['ExcNeuron' if x in ['L23', 'L4', 'L5', 'L6'] else x for x in lb]
pca_vec = np.load('/Users/ckw/warehouse/metacell/stark/test_output/pca_vec_1000000.npy')
umap_vec = np.load('/Users/ckw/warehouse/metacell/stark/test_output/umap_vec_1000000.npy')
cell_embeddings = pca_vec  # 或者 umap_vec，取决于你想用哪个作为输入
print(pca_vec.shape, umap_vec.shape, len(lb))
tmp_val = pca_vec.copy()
min_val = np.min(tmp_val)
tmp_val -= min_val
adata3 = ad.AnnData(tmp_val)
adata3.obs['cell_type'] = lb
adata3.obsm['X_pca'] = pca_vec
adata3.obsm['X_umap'] = umap_vec

(700, 123) (700, 2) 700


In [30]:
adata1.write('./view1.h5ad')
adata2.write('./view2.h5ad')
adata3.write('./view3.h5ad')

In [31]:
view1_adata = sc.read_h5ad('./view1.h5ad')
view2_adata = sc.read_h5ad('./view2.h5ad')
view3_adata = sc.read_h5ad('./view3.h5ad')

In [32]:
import scanpy as sc

for path in ["./view1.h5ad", "./view2.h5ad", "./view3.h5ad"]:
    adata = sc.read_h5ad(path)
    print(f"\n{path}: shape={adata.shape}")
    print(f"  NaN in X: {np.isnan(adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X).any()}")
    print(f"  obs keys: {list(adata.obs.keys())}")


./view1.h5ad: shape=(700, 114)
  NaN in X: False
  obs keys: ['cell_type']

./view2.h5ad: shape=(700, 68)
  NaN in X: False
  obs keys: ['cell_type']

./view3.h5ad: shape=(700, 123)
  NaN in X: False
  obs keys: ['cell_type']


In [33]:

run_metaq(
    data_path=[
        "./view1.h5ad",
        './view2.h5ad',
        './view3.h5ad'
    ],  # the path to the input h5ad data
    train_epoch=300,
    data_type=[
        "ADT","ADT","ADT",],  
    type_key='cell_type',
    # the type of the input data
    metacell_num=15,  # the target number of metacells
    save_name="view123",  # the file name prefix when saving the results
    device='cpu'
)


=======Loading and Preprocessing Data=======
Data of 3 omics in total
./view1.h5ad loaded with shape [700, 114]
./view2.h5ad loaded with shape [700, 68]
./view3.h5ad loaded with shape [700, 123]
Target metacell number: 15
======= Training Start =======
[Epoch 20] ADT: Loss Rec=6.5676 Loss Rec Q=6.6322 | ADT: Loss Rec=7.5790 Loss Rec Q=7.5687 | ADT: Loss Rec=6.7605 Loss Rec Q=6.7909 | Codebook: Loss C=0.2175
[Epoch 40] ADT: Loss Rec=6.2487 Loss Rec Q=6.1921 | ADT: Loss Rec=7.5135 Loss Rec Q=7.4987 | ADT: Loss Rec=6.4708 Loss Rec Q=6.4083 | Codebook: Loss C=0.2360
[Epoch 60] ADT: Loss Rec=5.5379 Loss Rec Q=5.4435 | ADT: Loss Rec=6.8701 Loss Rec Q=6.9657 | ADT: Loss Rec=5.8573 Loss Rec Q=5.6630 | Codebook: Loss C=0.2456
[Epoch 80] ADT: Loss Rec=4.9615 Loss Rec Q=4.8480 | ADT: Loss Rec=6.3513 Loss Rec Q=6.5118 | ADT: Loss Rec=5.3207 Loss Rec Q=5.0784 | Codebook: Loss C=0.2649
[Epoch 100] ADT: Loss Rec=4.5230 Loss Rec Q=4.6784 | ADT: Loss Rec=5.8755 Loss Rec Q=6.2352 | ADT: Loss Rec=4.8619 

In [57]:
adata = sc.read_h5ad('./save/view123_15metacell_ids.h5ad')
adata

AnnData object with n_obs × n_vars = 700 × 96
    obs: 'metacell', 'cell_type'
    uns: 'cell_type_colors', 'neighbors', 'umap'
    obsm: 'X_umap'
    obsp: 'connectivities', 'distances'

In [58]:
adata = sc.read_h5ad('./save/view123_15metacell_ids.h5ad')
adata.obs['label'] = lb
adata.uns['X_pca'] = pca_vec    
adata.uns['X_umap'] = umap_vec

In [59]:
hdata = sk.create_hdata_from_adata(adata,
                                 data_dir="/Users/ckw/warehouse/metacell/data/test_700_snm3c",
                                output_dir="/Users/ckw/warehouse/metacell/stark/test_output",
                                genome_reference_path="/Users/ckw/warehouse/metacell/hg19.fa.chrom.sizes",
                                chrom_list=[f"chr{i}" for i in range(1, 23)],
                                resolution=[500000])
hdata

HData object with 700 cells and 0 metacells
    resolutions: [500000]
    obs: ['metacell', 'cell_type', 'label']
    views_pca: [500000]
    views_umap: [500000]
    views_embedding: []
    views_mat: []
    views_is: []
    uns keys: []

In [60]:
purity_df, metrics = sk.tl.evaluate(hdata, hdata.obs['label'])


正在计算评估指标...
✅ 指标计算完成！(发现 11 种细胞类型)
----------------------------------------
简单平均纯度 (Mean Purity)  : 0.4666
模型准确率 (Accuracy)      : 0.3143
全局加权分 (Global Score)  : 0.1363
过度融合指标 (WCOS)       : 0.5686
Hub 权重不纯度 (HWIS)     : 0.7514
----------------------------------------
✅ 评估指标计算完成，纯度得分(EP_v2等)已同步至 hdata.metacells。


In [66]:
np.array(list(metrics.values()))

array([0.46662954, 0.31428571, 0.13633274, 0.56857143, 0.75140204,
       0.52627702, 0.22175862])

In [65]:
np.array(list(metrics_summary.values()))

array([0.52627702, 0.22175862])

In [62]:
res_df, metrics_summary = sk.tl.evaluate_metacell(
    hdata=hdata,
    use_view=500000,      # 默认选第0个组学视角，也可以传入字典里的特定 key
    metric='euclidean'  # 或 'cosine'
)


正在计算 Metacell 空间结构指标 (Compactness & Separation)...
使用的特征视图: 500000 (距离度量: euclidean)
----------------------------------------
全局平均紧凑度得分 (Compactness Score): 0.5263 (值域 0~1，1=最紧凑/最佳)
全局平均分离度得分 (Separation Score) : 0.2218 (值域 0~1，1=最分离/最佳)
----------------------------------------
✅ 空间分布评估指标计算完成，得已并入 hdata.metacells 及 hdata.uns 缓存中。


In [ ]:
hdata

HData object with 4236 cells and 40 metacells
    resolutions: [500000]
    obs: ['metacell', 'cell_type', 'label']
    views_pca: [500000]
    views_umap: [500000]
    views_embedding: []
    views_mat: []
    views_is: []
    uns keys: ['purity_df', 'metrics', 'eval_df_cache', 'avg_size_cache', 'thre_cache', 'accuracy', 'global_score', 'wcos', 'hwis']
    metacells: ['CellType', 'CellType_purity', 'cell_num', 'w_min', 'w_max', 'P_adj', 'EP_v2']
    metacell_data keys: []